In [ ]:
import pandas as pd
import io
import base64
import re
import torch
from PIL import Image
from tqdm import tqdm
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# ==================== 1. 答案清洗器 ====================
def extract_option(model_output, true_answer):
    clean_output = model_output.strip().upper()
    if not clean_output: return "N/A"
    if clean_output in ['A', 'B', 'C', 'D']: return clean_output
    match = re.search(r'\b([A-D])\b', clean_output)
    if match: return match.group(1)
    true_ans_upper = str(true_answer).strip().upper()
    if true_ans_upper in clean_output:
        other_options = [opt for opt in ['A', 'B', 'C', 'D'] if opt != true_ans_upper]
        if not any(opt in clean_output for opt in other_options):
            return true_ans_upper
    pure_chars = re.sub(r'[^\w]', '', clean_output)
    if pure_chars and pure_chars[0] in ['A', 'B', 'C', 'D']: return pure_chars[0]
    return "N/A"

# ==================== 2. 环境与模型初始化 ====================
model_path = "./model_weights/qwen/Qwen3-VL-2B-Instruct"
input_file = "MMBench_DEV_EN.tsv"
output_file = "MMBench_5060_Final_Results.csv"

print("📦 正在为您唤醒 5060 纯GPU原生引擎...")

# 🌟 优化点 1：从底层限制视觉 Token 数量，杜绝大图导致的 prefill 耗时
# Qwen-VL 系列模型以 28*28 像素为一个视觉标记
min_pixels = 224 * 28 * 28  # 最小
max_pixels = 448 * 28 * 28  # 最大极限
processor = AutoProcessor.from_pretrained(model_path, min_pixels=min_pixels, max_pixels=max_pixels)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    attn_implementation="sdpa"  # 矩阵硬件级加速
)
model.eval()

# 🌟 优化点 2：在后台悄悄完成 CUDA 热身，不让垃圾时间污染你的进度条！
print("🔥 正在为 RTX 5060 进行核心热身 (CUDA Warm-up)...")
with torch.no_grad():
    dummy_image = Image.new('RGB', (224, 224), color='white')
    dummy_text = "<|im_start|>user\n<|image_pad|>Question: Is it white?\nA. Yes\nB. No\nAnswer directly.<|im_end|>\n<|im_start|>assistant\n"
    dummy_inputs = processor(text=[dummy_text], images=[dummy_image], return_tensors="pt").to(model.device)
    _ = model.generate(**dummy_inputs, max_new_tokens=2)
print("✅ 热身完毕！显卡核心频率已拉满。")

# ==================== 3. 读取数据 ====================
df = pd.read_csv(input_file, sep="\t")
total_questions = len(df)

all_results = []
correct_count = 0
processed_count = 0

print(f"🚀 5060 纯净单例全速评测启动...")

# ==================== 4. 纯显存单例循环 ====================
for idx, row in tqdm(df.iterrows(), total=total_questions, desc="5060 真实算力疾驰中"):
    try:
        image_b64 = row['image']
        image_obj = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
        image_obj.thumbnail((448, 448)) 
        
        question = row['question']
        options = f"\nA. {row['A']}\nB. {row['B']}"
        if pd.notna(row.get('C')): options += f"\nC. {row['C']}"
        if pd.notna(row.get('D')): options += f"\nD. {row['D']}"
        hint = f"Hint: {row['hint']}\n" if pd.notna(row.get('hint')) else ""
        
        prompt_text = f"{hint}Question: {question}{options}\nAnswer with the option letter directly."
        
        messages = [{"role": "user", "content": [{"type": "image", "image": image_obj}, {"type": "text", "text": prompt_text}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        inputs = processor(text=[text], images=[image_obj], return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=10)
            
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        model_output = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
        
        true_answer = str(row['answer']).strip().upper()
        pred_char = extract_option(model_output, true_answer)
        
        is_correct = (pred_char == true_answer)
        if is_correct: correct_count += 1
        processed_count += 1
        
        all_results.append({
            "index": row['index'],
            "true_answer": true_answer,
            "pred_answer": pred_char,
            "is_correct": is_correct,
            "model_output_raw": model_output
        })
    except Exception as e:
        continue

# ==================== 5. 战报 ====================
final_accuracy = (correct_count / processed_count) * 100 if processed_count > 0 else 0
print("\n" + "="*50)
print(f"📈 最终总准确率 (Total Accuracy): {final_accuracy:.2f}%")
print(f"✅ 真实答对题目数: {correct_count} / {processed_count}")
print("="*50)

df_results = pd.DataFrame(all_results)
df_results.to_csv(output_file, index=False, encoding="utf-8-sig")

📦 正在为您唤醒 5060 纯GPU原生引擎...


Loading weights: 100%|██████████| 625/625 [00:01<00:00, 424.75it/s]


🔥 正在为 RTX 5060 进行核心热身 (CUDA Warm-up)...
✅ 热身完毕！显卡核心频率已拉满。
🚀 5060 纯净单例全速评测启动...


5060 真实算力疾驰中:   0%|          | 9/4329 [01:09<9:14:35,  7.70s/it] 


KeyboardInterrupt: 